## Cómo Ejecutar Este Notebook

**Orden de Ejecución (importante seguir este orden):**

1. **Celda 2**: Imports y configuración de pandas
2. **Celda 5**: Función `etl_transform()` - Pipeline de transformación
3. **Celda 6**: Función `load_and_process_years()` - Orquestador multi-año
4. **Celda 7** (opcional): Diagnóstico de estructura de archivos
5. **Celda 8**: **Ejecutar ETL** - Procesa los 3 años (2019, 2020, 2021)
6. **Celda 9**: **Exportar datasets** - Guarda archivos limpios en `/data/clean/`
7. **Celdas 10-13**: Validación del dataset unificado

**Resultado esperado:**
- `historico_2019_2021_clean.csv` - Dataset unificado (~16M registros)
- `historico_2019_clean.csv` - Dataset individual 2019
- `historico_2020_clean.csv` - Dataset individual 2020
- `historico_2021_clean.csv` - Dataset individual 2021

---

# Etapa 2 – Limpieza y Transformación (ETL)
Proyecto: Subtes de Buenos Aires

Objetivo: asegurar calidad de datos (nulos, duplicados, tipos, consistencia), transformar variables necesarias y dejar un dataset final listo para visualización (Etapa 3).


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # muestra todas las columnas
pd.set_option("display.width", 120) # ajusto el ancho


## Metodología: ETL Reutilizable Multi-Año

**Decisión arquitectónica:**  
En lugar de duplicar el proceso ETL por cada año, decidi implementar un **pipeline funcional reutilizable** que garantiza:

- **Consistencia**: transformaciones idénticas para todos los años
- **Mantenibilidad**: cambios en un solo lugar
- **Comparabilidad**: datos estandarizados entre períodos (pre-pandemia, pandemia, post-pandemia)

**Años a procesar:**
- 2019 (pre-pandemia)
- 2020 (pandemia)
- 2021 (post-pandemia)

## Función ETL Reutilizable

Encapsula todas las transformaciones en una función que recibe un DataFrame y retorna el DataFrame procesado.

In [4]:
# Defino los años a procesar
years_to_process = [2019, 2020, 2021]
path = "../data/raw/"

# Ejecutar ETL multi-año
df_completo = load_and_process_years(years_to_process, path)


Cargando historico_2019.csv...
   - 12,662,343 registros cargados

Procesando año 2019
Columnas encontradas: ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
- Registros con inconsistencia en 'total': 0
 OJO! 7,646,139 registros con fechas inválidas serán eliminados
PERFECTO: Conversión de fecha completada
PERFECTO: Optimización de columnas numéricas completada
PERFECTO: Normalización de textos completada
- Duplicados eliminados: 0
 Extracción de horas completada
- Variables derivadas creadas

Resumen final 2019:
  - Registros: 5,016,204
  - Memoria: 1005.29 MB
  - Rango fechas: 2019-01-01 00:00:00 a 2019-12-12 00:00:00

Cargando historico_2020.csv...
   - 5,781,006 registros cargados
   → Columna 'periodo' generada desde 'fecha'

Procesando año 2020
Columnas encontradas: ['fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total', 'periodo']
- Registros c

In [5]:
# Exportación del dataset completo
path_clean = "../data/clean/"

# Export unificado
df_completo.to_csv(path_clean + "historico_2019_2021_clean.csv", index=False)
print(f"- Dataset completo exportado: historico_2019_2021_clean.csv")

# Exports individuales por año (opcional, para análisis específicos)
for year in df_completo['anio'].unique():
    df_year = df_completo[df_completo['anio'] == year]
    df_year.to_csv(path_clean + f"historico_{year}_clean.csv", index=False)
    print(f"- Dataset {year} exportado: historico_{year}_clean.csv")

# Verificación post-exportación
df_test = pd.read_csv(path_clean + "historico_2019_2021_clean.csv", nrows=5)
print(f"\n- Verificación exitosa - {len(df_test)} registros leídos como prueba")

- Dataset completo exportado: historico_2019_2021_clean.csv
- Dataset 2019 exportado: historico_2019_clean.csv
- Dataset 2020 exportado: historico_2020_clean.csv
- Dataset 2021 exportado: historico_2021_clean.csv

- Verificación exitosa - 5 registros leídos como prueba


## Exportación del Dataset Unificado

In [ ]:
# Verificación de nulos
print("Valores nulos por columna:")
print(df_completo.isnull().sum())

In [ ]:
# Distribución de registros por año
print("Distribución de registros por año:")
print(df_completo.groupby('anio').size())
print("\nEstadísticas por año:")
print(df_completo.groupby('anio')['total'].describe())

In [ ]:
# Información detallada
df_completo.info(memory_usage="deep")

In [ ]:
# Vista general del dataset
df_completo.head()

## Validación del Dataset Unificado

## Procesamiento Multi-Año

Aplicamos el ETL a los años 2019, 2020 y 2021 de forma consistente.

In [3]:
def load_and_process_years(years, path_raw="../data/raw/"):
    """
    Carga y procesa múltiples años de datos históricos.
    
    Parametros:
    -----------
    years : list
        Lista de años a procesar (ej: [2019, 2020, 2021])
    path_raw : str
        Ruta a los archivos CSV en crudos, osea raw
    
    Returns:
    --------
    pd.DataFrame
        DataFrame concatenado con todos los años procesados
    """
    dfs_procesados = []
    
    for year in years:
        filename = f"historico_{year}.csv"
        filepath = path_raw + filename
        
        print(f"\nCargando {filename}...")
        try:
            # Intentar cargar con detección automática de separador
            df = pd.read_csv(filepath)
            
            # Si solo tiene 1 columna, intentar con separador ;
            if len(df.columns) == 1:
                df = pd.read_csv(filepath, sep=';')
                print(f"   -> Archivo cargado con separador ';'")
            
            print(f"   - {len(df):,} registros cargados")
            
            # Normalizar nombres de columnas a minúsculas
            df.columns = df.columns.str.lower().str.strip()
            
            # Renombrar columnas inconsistentes
            column_mapping = {
                'pax_total': 'total',
            }
            df.rename(columns=column_mapping, inplace=True)
            
            # Si falta 'periodo', lo creo desde 'fecha'
            if 'periodo' not in df.columns and 'fecha' in df.columns:
                # Parsear fechas con dayfirst=True para manejo el formato dd/mm/yyyy
                df['periodo'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce').dt.to_period('M').astype(str).str.replace('-', '')
                print(f"   → Columna 'periodo' generada desde 'fecha'")
            
            # Aplicar ETL
            df_clean = etl_transform(df, year)
            dfs_procesados.append(df_clean)
            
        except FileNotFoundError:
            print(f"  ERROR: Archivo no encontrado: {filepath}")
        except Exception as e:
            print(f"   Error procesando {year}: {e}")
            import traceback
            traceback.print_exc()
    
    # Concatenar todos los DataFrames en uno
    if dfs_procesados:
        print(f"\n{'='*60}")
        print("Unificando datasets...")
        print(f"{'='*60}")
        
        df_final = pd.concat(dfs_procesados, ignore_index=True)
        
        print(f"\n DATASET UNIFICADO CREADO:")
        print(f"  - Total registros: {len(df_final):,}")
        print(f"  - Años incluidos: {sorted(df_final['anio'].unique())}")
        print(f"  - Memoria total: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        print(f"  - Rango completo: {df_final['fecha'].min()} a {df_final['fecha'].max()}")
        
        return df_final
    else:
        print("\nATENCION: No se pudo procesar ningún archivo")
        return None

In [2]:
def etl_transform(df, year):
    """
    Aplica el pipeline completo de transformación ETL a un DataFrame de historico de subtes.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame cargado desde el CSV raw
    year : int
        Año del dataset (para logging y validación)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame transformado y optimizado
    """
    print(f"\n{'='*60}")
    print(f"Procesando año {year}")
    print(f"{'='*60}")
    
    # Verificar columnas presentes
    print(f"Columnas encontradas: {list(df.columns)}")
    
    # Columnas esperadas
    expected_cols = ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 
                     'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
    
    missing_cols = [col for col in expected_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Columnas faltantes: {missing_cols}")
    
    # 1. Validación de integridad: total = suma de componentes
    df['total_calculado'] = (
        df['pax_pagos'] + 
        df['pax_pases_pagos'] + 
        df['pax_franq']
    )
    
    inconsistencias = (df['total'] != df['total_calculado']).sum()
    print(f"- Registros con inconsistencia en 'total': {inconsistencias}")
    
    if inconsistencias > 0:
        print("  -> Corrigiendo inconsistencias...")
        df['total'] = df['total_calculado']
    
    df.drop('total_calculado', axis=1, inplace=True)
    
    # 2. Conversión de fechas (con dayfirst=True para manejar dd/mm/yyyy)
    df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
    
    # Filtrar registros con fechas inválidas
    fechas_invalidas = df['fecha'].isnull().sum()
    if fechas_invalidas > 0:
        print(f" OJO! {fechas_invalidas:,} registros con fechas inválidas serán eliminados")
        df = df[df['fecha'].notnull()].reset_index(drop=True)
    
    print(f"PERFECTO: Conversión de fecha completada")
    
    # 3. Optimización de tipos - periodo (manejar valores no numéricos)
    # Convierto el periodo a numérico, forzando errores a NaN
    df["periodo"] = pd.to_numeric(df["periodo"], errors='coerce')
    
    # Elimino registros con periodo inválido
    periodo_invalido = df["periodo"].isnull().sum()
    if periodo_invalido > 0:
        print(f" ERROR: {periodo_invalido:,} registros con periodo inválido serán eliminados")
        df = df[df["periodo"].notnull()].reset_index(drop=True)
    
    df["periodo"] = df["periodo"].astype("int32")
    
    # 4. Optimización de columnas numéricas
    for col in ["pax_pagos", "pax_pases_pagos", "pax_franq", "total"]:
        max_val = df[col].max()
        if max_val <= 255:
            df[col] = df[col].astype("uint8")
        elif max_val <= 65535:
            df[col] = df[col].astype("uint16")
        else:
            df[col] = df[col].astype("uint32")
    
    print(f"PERFECTO: Optimización de columnas numéricas completada")
    
    # 5. Normalización de textos
    df["estacion"] = (
        df["estacion"].astype("string")
        .str.strip()
        .str.title()
        .astype("category")
    )
    
    df["linea"] = (
        df["linea"].astype("string")
        .str.strip()
        .str.upper()
        .astype("category")
    )
    
    df["molinete"] = (
        df["molinete"].astype("string")
        .str.strip()
        .str.upper()
        .astype("category")
    )
    
    print(f"PERFECTO: Normalización de textos completada")
    
    # 6. Eliminación de duplicados
    duplicados_inicial = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicados_eliminados = duplicados_inicial - len(df)
    print(f"- Duplicados eliminados: {duplicados_eliminados}")
    
    # 7. Variables derivadas
    try:
        df["hora_desde"] = pd.to_datetime(
            df["desde"], 
            format='%H:%M:%S', 
            errors='coerce'
        ).dt.hour.astype("Int8")
        
        df["hora_hasta"] = pd.to_datetime(
            df["hasta"], 
            format='%H:%M:%S', 
            errors='coerce'
        ).dt.hour.astype("Int8")
        
        print(f" Extracción de horas completada")
    except Exception as e:
        print(f" No se pudieron procesar las horas: {e}")
    
    # Variables temporales
    df["dia_semana"] = df["fecha"].dt.day_name()
    df["mes"] = df["fecha"].dt.month.astype("Int8")
    df["dia_mes"] = df["fecha"].dt.day.astype("Int8")
    df["es_fin_semana"] = df["dia_semana"].isin(['Saturday', 'Sunday']).astype("int8")
    df["anio"] = year  # Agregar columna de año para identificación
    
    print(f"- Variables derivadas creadas")
    
    # Resumen final
    memoria_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\nResumen final {year}:")
    print(f"  - Registros: {len(df):,}")
    print(f"  - Memoria: {memoria_mb:.2f} MB")
    print(f"  - Rango fechas: {df['fecha'].min()} a {df['fecha'].max()}")
    
    return df

In [ ]:
# Diagnóstico rápido de estructura de archivos
path = "../data/raw/"

for year in [2019, 2020, 2021]:
    print(f"\n{'='*60}")
    print(f"Estructura de historico_{year}.csv")
    print(f"{'='*60}")
    
    df_test = pd.read_csv(path + f"historico_{year}.csv", nrows=5)
    print(f"Columnas: {list(df_test.columns)}")
    print(f"Tipos de datos:")
    print(df_test.dtypes)
    print(f"\nPrimeras filas:")
    print(df_test.head(2))

## Diagnóstico de Estructura de Archivos

Antes de aplicar el ETL, verificamos la estructura de cada archivo para identificar diferencias.

## Conclusiones de la Etapa 2 - ETL Multi-Año

### Decisión Arquitectónica
Implementamos un **pipeline ETL funcional y reutilizable** que garantiza:
- **Consistencia**: transformaciones idénticas aplicadas a 2019, 2020 y 2021
- **Mantenibilidad**: toda la lógica ETL centralizada en funciones
- **Escalabilidad**: fácil agregar nuevos años modificando solo la lista `years_to_process`
- **Comparabilidad**: datos estandarizados para análisis temporal pre/durante/post-pandemia

### Transformaciones Aplicadas
- Validación de integridad (`total = pax_pagos + pax_pases_pagos + pax_franq`)
- Conversión de tipos (datetime, categorías, optimización numérica uint8/uint16/uint32)
- Normalización de textos (Title/Upper case consistente)
- Eliminación de duplicados
- Variables derivadas (hora_desde, hora_hasta, dia_semana, mes, dia_mes, es_fin_semana)
- Columna `anio` para identificación y segmentación temporal

### Dataset Resultante
**Archivo unificado:** `historico_2019_2021_clean.csv`
- Años: 2019, 2020, 2021
- Calidad: sin nulos, sin duplicados, tipos optimizados
- Memoria optimizada mediante casting inteligente

**Archivos individuales:** `historico_YYYY_clean.csv`
- Disponibles para análisis específicos por año

### Listo para Etapa 3 (Visualización)
El dataset está preparado para análisis comparativo:
- **Impacto pandemia**: comparación 2019 vs 2020 vs 2021
- **Patrones temporales**: horarios pico, días semana, estacionalidad
- **Distribución espacial**: análisis por líneas y estaciones
- **Composición de pasajeros**: pagos vs franquicias por período

### Ventajas del Enfoque Funcional
1. **Un solo lugar de mantenimiento**: cambios en el ETL se aplican automáticamente a todos los años
2. **Trazabilidad**: logs detallados por año durante el procesamiento
3. **Validación consistente**: mismas reglas de calidad en todos los períodos
4. **Reproducibilidad**: ejecutar el notebook genera resultados idénticos